# Cursor Remote SSH + CUDA workbook

## How to run

1. Add a real Linux/NVIDIA host to `~/.ssh/config`.
2. In Cursor, run `Remote-SSH: Connect to Host...` and choose that host.
3. Open the remote checkout of this project.
4. In the remote terminal, run `uv sync` and `nvidia-smi`.
5. Open this notebook. In the notebook toolbar above the first cell, click `Select Kernel` (or the current kernel name).
6. Choose `Select Another Kernel...`, then `Python Environments...`, then choose the remote `.venv/bin/python` environment.
7. Click `Run All` in that same notebook toolbar.

The first code cell starts the kernel on the remote Linux/NVIDIA machine. Every cell executes there. There is intentionally no CPU or MPS fallback.

![Cursor notebook kernel picker flow](../docs/images/cursor-kernel-picker-flow.svg)

The flow above matches Cursor's current UI. Do not use the Command Palette's `Jupyter: Manage Access To Jupyter Kernels`. The remote `.venv/bin/python` remains inside the remote project folder.

In [ ]:
import platform

import torch

assert torch.cuda.is_available(), "This notebook requires an NVIDIA CUDA runtime."
device = torch.device("cuda")
print(
    {
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(0),
    }
)

In [ ]:
left = torch.ones((1024, 1024), device=device)
right = torch.full((1024, 1024), 2.0, device=device)
product = left @ right
torch.cuda.synchronize()
print(
    {
        "device": str(product.device),
        "shape": tuple(product.shape),
        "sample": float(product[0, 0].cpu()),
    }
)

In [ ]:
weights = torch.randn(1024, 1024, device=device, requires_grad=True)
loss = weights.square().mean()
loss.backward()
torch.cuda.synchronize()
print({"loss": float(loss), "gradient_norm": float(weights.grad.norm())})